## Library Installation

In [1]:
# update or install the necessary libraries

!pip install --upgrade langchain google-genai langchain-google-genai
# !pip install --upgrade langchain openai langchain-openai
!pip install --upgrade python-dotenv

## Code

In [2]:
import os
import IPython

In [3]:
from google.colab import userdata

In [4]:
from dotenv import load_dotenv

In [5]:
from langchain.messages import HumanMessage, SystemMessage
# from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

In [6]:
# Directly set the API key
# api_key = userdata.get('OPENAI_API_KEY')
# os.environ["OPENAI_API_KEY"] = api_key
api_key = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = api_key

In [7]:
def set_params(
    model="gemma-4-31b-it",
    temperature=1,
    max_tokens=256,
    frequency_penalty=0,
    presence_penalty=0,
):
    """ set ai parameters"""

    return {
        'model': model,
        'temperature': temperature,
        'max_tokens': max_tokens,
        'frequency_penalty': frequency_penalty,
        'presence_penalty': presence_penalty,
    }

In [8]:
def get_completion(params, messages):
    """ GET completion from ai api"""
    model = ChatGoogleGenerativeAI(
        model=params['model'],
        temperature=params['temperature'],
        max_output_tokens=params['max_tokens'],
        frequency_penalty=params['frequency_penalty'],
        presence_penalty=params['presence_penalty'],
    )
    response = model.invoke(messages)
    return response

In [9]:
# basic example
params = set_params()

prompt = "The sky is"

messages = [
    SystemMessage(content="Please complete the sentence."),
    HumanMessage(content=prompt)
]

In [10]:
messages

[SystemMessage(content='Please complete the sentence.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='The sky is', additional_kwargs={}, response_metadata={})]

In [11]:
response = get_completion(params, messages)

In [12]:
response.text

'blue.'

Try with different temperature to compare results:

In [13]:
params = set_params(temperature=0)
response = get_completion(params, messages)
IPython.display.Markdown(response.text)

blue.

### 1.1 Text Summarization

In [14]:
# Updated example with a different topic
params = set_params(temperature=1, max_tokens=400)
prompt = """Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy into chemical energy.
They use sunlight, carbon dioxide, and water to produce glucose and oxygen, which are essential for the survival of most life forms on Earth.

Explain the above in one sentence:"""

messages = [
    HumanMessage(content=prompt)
]

response = get_completion(params, messages)

# Display the result in Markdown format
IPython.display.Markdown(response.text)

Photosynthesis is the process by which green plants, algae, and some bacteria convert sunlight, water, and carbon dioxide into glucose and oxygen, sustaining most life on Earth.

### 1.2 Question Answering

In [15]:
prompt = """Answer the question based on the context below. Keep the answer short. Respond "Unsure about answer" if not sure.

Context: Alexander Fleming discovered penicillin in 1928 when he noticed that a mold called Penicillium notatum could kill bacteria.

Question: What was the name of the mold that led to the discovery of penicillin?

Answer:"""

In [16]:
messages = [
    HumanMessage(content=prompt)
]

response = get_completion(params, messages)
IPython.display.Markdown(response.text)

Penicillium notatum

### 1.3 Text Classification

In [17]:
prompt = """Classify the text into neutral, negative, or positive.

Text: The customer service was absolutely terrible.

Sentiment:"""

messages = [
    HumanMessage(content=prompt)
]

response = get_completion(params, messages)
IPython.display.Markdown(response.text)

Negative

### 1.4 Role Playing

In [18]:
prompt = """The following is a conversation with an AI research assistant. The assistant's tone is technical and scientific.

Human: Hello, who are you?
AI: Greetings! I am an AI research assistant. How can I assist you today?
Human: Can you explain the process of quantum entanglement?
AI:"""

messages = [
    HumanMessage(content=prompt)
]

response = get_completion(params, messages)
IPython.display.Markdown(response.text)

Quantum entanglement is a physical phenomenon

### 1.5 Code Generation

In [19]:
prompt = "\"\"\"\nTable employees, columns = [EmployeeId, EmployeeName, DepartmentId]\nTable departments, columns = [DepartmentId, DepartmentName]\nCreate a MySQL query to retrieve all employees working in the Finance department.\n\"\"\""

messages = [
    HumanMessage(content=prompt)
]

response = get_completion(params, messages)
IPython.display.Markdown(response.text)

```sql
SELECT e.EmployeeName
FROM employees e
JOIN departments d ON e.DepartmentId = d.DepartmentId
WHERE d.DepartmentName = 'Finance';
```

**Explanation:**
*   **`SELECT e.EmployeeName`**: Chooses the column we want to retrieve from the employees table.
*   **`FROM employees e`**: Sets the starting table and gives it an alias (`e`) for easier referencing.
*   **`JOIN departments d ON e.DepartmentId = d.DepartmentId`**: Performs an inner join to link the two tables based on

### 1.6 Reasoning

In [20]:
prompt = """The even numbers in this group add up to an odd number: 14, 27, 8, 19, 6, 11, 2.

Solve by breaking the problem into steps. First, identify the even numbers, add them, and indicate whether the result is odd or even."""

messages = [
    HumanMessage(content=prompt)
]

response = get_completion(params, messages)
IPython.display.Markdown(response.text)

Here are the steps to solve the problem:

1.  **Identify the even numbers:** 14, 8, 6, 2
2.  **Add them together:** 14 + 8 + 6 + 2 = 30
3.  **Determine if the result is odd or even:** 30 is an

### 2.2 Few-shot prompts

In [21]:
prompt = """The even numbers in this group add up to an odd number: 3, 6, 8, 10, 15, 2.
A: The answer is False.

The even numbers in this group add up to an odd number: 12, 7, 18, 20, 5, 14.
A: The answer is False.

The even numbers in this group add up to an odd number: 11, 22, 16, 4, 9, 8.
A: The answer is True.

The even numbers in this group add up to an odd number: 2, 5, 10, 14, 18, 7.
A: The answer is False.

The even numbers in this group add up to an odd number: 24, 32, 6, 10, 18, 4.
A:"""

messages = [
    HumanMessage(content=prompt)
]

response = get_completion(params, messages)
IPython.display.Markdown(response.text)

The answer is False.

### Chain-of-Thought (CoT)

In [22]:
prompt = """The even numbers in this group add up to an odd number: 3, 6, 8, 10, 15, 2.
A: Adding all the even numbers (6, 8, 10, 2) gives 26. The answer is False.

The even numbers in this group add up to an odd number: 12, 7, 18, 20, 5, 14.
A:"""

messages = [
    HumanMessage(content=prompt)
]

response = get_completion(params, messages)
IPython.display.Markdown(response.text)

Adding all the even numbers (12, 18, 20, 14) gives 64. The answer is False.

### Zero-shot CoT

In [23]:
prompt = """I had 20 pencils. I gave 4 pencils to my friend and 3 to my teacher. Later, I bought 6 more pencils and lost 2. How many pencils do I have now?

Let's think step by step."""

messages = [
    HumanMessage(content=prompt)
]

response = get_completion(params, messages)
IPython.display.Markdown(response.text)


Here is the step-by-step breakdown:

1.  **Starting amount:** 20 pencils
2.  **Give 4 to a friend:** 20 - 4 = 16 pencils
3.  **Give 3 to a teacher:** 16 - 3 = 13 pencils
4.  **Buy 6 more:** 13 + 6 = 19 pencils
5.  **Lose 2:** 19 - 2 = 17 pencils

You

### Tree of Thought

In [24]:
prompt = """I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman.
I then went and bought 5 more apples and ate 1. How many apples do I have now?

Let's think step by step using a tree of thought approach:

### Step 1: Identify possible solution paths
1. **Path A (Subtraction First Approach)**
   - Start with 10 apples.
   - Remove apples given away first, then add new apples.
   - Finally, subtract the one eaten.

2. **Path B (Addition First Approach)**
   - Start with 10 apples.
   - Add newly purchased apples first, then subtract those given away.
   - Finally, subtract the one eaten.

### Step 2: Solve for each path
Let's compute the total apples for both paths and compare results.

Final Answer:"""

messages = [
    HumanMessage(content=prompt)
]

response = get_completion(params, messages)
IPython.display.Markdown(response.text)


**Path A (Subtraction First Approach):**
1. Start with **10** apples.
2. Give 2 to the neighbor and 2 to the repairman: $10 - 2 - 2 = 6$ apples.
3. Buy 5 more apples: $6 + 5 = 11$ apples.
4. Eat 1 apple: $11 - 1 = 10$ apples.

**Path B (Addition First Approach):**
1. Start with **10** apples.
2. Add the 5 newly purchased apples: $10 +